# Additional Ablations

1. **Hybrid candidate pool size** — does a larger `candidate_k` improve MRR?
2. **BM25 tokenization** — effect of stopword removal and stemming
3. **Query length analysis** — does Dense / BM25 dominance depend on query length?
4. **Approximate FAISS (IndexIVFFlat)** — quality vs. speed tradeoff for large corpora
5. **BM25 k1=0.5 inside Hybrid** — does the better k1 carry over to the Hybrid?


In [ ]:
import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import faiss
import torch
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

from loader import load_data
from retrievers.bm25 import BM25Retriever
from retrievers.tf_idf_retriever import TFIDFRetriever
from retrievers.dense_retriever import DenseRetriever
from retrievers.hybrid_retriever import HybridRetriever
from encoders.sbert import SentenceBERTEncoder
from evaluation.mrr import mrr_at_10
from evaluation.timing import measure_retrieval_time

N = 50_000
TOP_K = 10

ds = load_data(n=N)
passages_text, queries = [], []
for example in ds:
    queries.append(example["query"])
    for p in example["passages"]["passage_text"]:
        passages_text.append(p)
print(f"Passages: {len(passages_text)}, Queries: {len(queries)}")

## 1. Hybrid Candidate Pool Size

The Hybrid fetches `candidate_k` results from each retriever before reranking.
A larger pool gives more recall but increases latency.

In [ ]:
results_pool = []

for ck in [10, 25, 50, 100, 200]:
    hybrid = HybridRetriever(top_k=TOP_K, candidate_k=ck)
    hybrid.fit(passages_text)
    mrr = mrr_at_10(hybrid, ds)
    t   = measure_retrieval_time(hybrid, queries)
    results_pool.append({"candidate_k": ck, "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"candidate_k={ck:<4}  MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del hybrid; gc.collect()

pd.DataFrame(results_pool)

## 2. BM25 Tokenization Strategies

The default `CountVectorizer` already lowercases. Testing:
- **Baseline**: lowercase only (default)
- **+ Stopwords**: remove common English stopwords
- **+ Stemming**: Porter stemmer (reduces terms to root form)


In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stop_words = set(nltk_stopwords.words('english'))

def stemming_tokenizer(text):
    return [stemmer.stem(w) for w in text.lower().split() if w.isalpha()]

def build_bm25(vectorizer_kwargs, top_k=TOP_K, k1=1.5, b=0.75):
    """BM25Retriever with a custom CountVectorizer configuration."""
    class _BM25:
        def fit(self, passages):
            self.top_k = top_k
            vect = CountVectorizer(**vectorizer_kwargs)
            tf = vect.fit_transform(passages)
            n_docs = tf.shape[0]
            doc_len = np.asarray(tf.sum(axis=1)).ravel()
            avgdl = doc_len.mean()
            df = np.asarray((tf > 0).sum(axis=0)).ravel()
            idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)
            coo = tf.tocoo()
            row, col, data = coo.row, coo.col, coo.data.astype(np.float64)
            denom = data + k1 * (1.0 - b + b * doc_len[row] / avgdl)
            bm25_vals = idf[col] * data * (k1 + 1.0) / denom
            self.bm25_matrix = csr_matrix((bm25_vals, (row, col)), shape=tf.shape, dtype=np.float32)
            self.vectorizer = vect

        def query(self, query):
            q_vec = self.vectorizer.transform([query])
            scores = (q_vec @ self.bm25_matrix.T).toarray().ravel()
            part = np.argpartition(scores, -self.top_k)[-self.top_k:]
            return list(map(int, part[np.argsort(scores[part])[::-1]]))

        def query_batch(self, queries, chunk_size=64):
            results = []
            for start in range(0, len(queries), chunk_size):
                chunk = queries[start:start+chunk_size]
                q_vecs = self.vectorizer.transform(chunk)
                scores_mat = (q_vecs @ self.bm25_matrix.T).toarray()
                for scores in scores_mat:
                    part = np.argpartition(scores, -self.top_k)[-self.top_k:]
                    results.append(list(map(int, part[np.argsort(scores[part])[::-1]])))
            return results
    return _BM25()

In [ ]:
results_tok = []

tokenization_configs = [
    ("Baseline (lowercase)",       {}),
    ("+ Stopword removal",          {"stop_words": "english"}),
    ("+ Stemming (no stopwords)",   {"tokenizer": stemming_tokenizer, "token_pattern": None}),
    ("+ Stemming + stopwords",      {"tokenizer": lambda t: [stemmer.stem(w) for w in t.lower().split() if w.isalpha() and w not in stop_words], "token_pattern": None}),
]

for name, kwargs in tokenization_configs:
    r = build_bm25(kwargs)
    r.fit(passages_text)
    mrr = mrr_at_10(r, ds)
    t   = measure_retrieval_time(r, queries)
    results_tok.append({"Tokenization": name, "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"{name:<35} MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del r; gc.collect()

pd.DataFrame(results_tok)

## 3. Query Length Analysis

Does Dense dominate BM25 equally across short and long queries?
Buckets: short (1–3 words), medium (4–6 words), long (7+ words).

In [ ]:
def mrr_by_length(retriever, dataset, min_len, max_len, max_queries=2000):
    items = []
    global_counter = 0
    for example in dataset:
        is_selected = example["passages"]["is_selected"]
        base = global_counter
        global_counter += len(is_selected)
        if 1 not in is_selected:
            continue
        q_len = len(example["query"].split())
        if min_len <= q_len <= max_len:
            items.append((example["query"], base + is_selected.index(1)))
        if len(items) >= max_queries:
            break
    if not items:
        return 0.0, 0
    qs = [q for q, _ in items]
    gids = [g for _, g in items]
    all_top_k = retriever.query_batch(qs) if hasattr(retriever, "query_batch") else [retriever.query(q) for q in qs]
    mrr = sum(1/(tk.index(g)+1) for g, tk in zip(gids, all_top_k) if g in tk)
    return mrr / len(items), len(items)

In [ ]:
print("Fitting retrievers for length analysis...")
bm25_r  = BM25Retriever(top_k=TOP_K)
dense_r = DenseRetriever(top_k=TOP_K)
bm25_r.fit(passages_text)
dense_r.fit("sbert_embeddings.npy", passages_text)

buckets = [("Short (1-3)", 1, 3), ("Medium (4-6)", 4, 6), ("Long (7+)", 7, 999)]
results_len = []

for label, lo, hi in buckets:
    for name, r in [("BM25", bm25_r), ("Dense", dense_r)]:
        mrr, count = mrr_by_length(r, ds, lo, hi)
        results_len.append({"Bucket": label, "Retriever": name, "MRR@10": round(mrr, 4), "Queries": count})
        print(f"{label:<15} {name:<6} MRR: {mrr:.4f}  (n={count})")

del bm25_r, dense_r; gc.collect()
pd.DataFrame(results_len).pivot(index="Bucket", columns="Retriever", values="MRR@10")

## 4. Approximate FAISS (IndexIVFFlat)

`IndexIVFFlat` partitions vectors into `nlist` clusters. At query time only `nprobe` clusters are searched — trading accuracy for speed.

Tests `nprobe` ∈ {1, 4, 16, 64} vs. the exact `IndexFlatIP` baseline.

In [ ]:
encoder = SentenceBERTEncoder()
doc_embs = np.load("sbert_embeddings.npy").astype(np.float32)
doc_embs /= np.linalg.norm(doc_embs, axis=1, keepdims=True)
dim = doc_embs.shape[1]

def make_ivf_retriever(index, normalize_q=True, top_k=TOP_K):
    class _R:
        def query(self, q):
            q_emb = encoder.encode([q]).astype(np.float32)
            if normalize_q:
                q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
            _, idx = index.search(q_emb, top_k)
            return idx[0].tolist()
        def query_batch(self, qs):
            q_embs = encoder.encode(qs).astype(np.float32)
            if normalize_q:
                q_embs /= np.linalg.norm(q_embs, axis=1, keepdims=True)
            _, idxs = index.search(q_embs, top_k)
            return idxs.tolist()
    return _R()

results_faiss = []

# Exact baseline
idx_flat = faiss.IndexFlatIP(dim)
idx_flat.add(doc_embs)
r = make_ivf_retriever(idx_flat)
mrr = mrr_at_10(r, ds)
t   = measure_retrieval_time(r, queries)
results_faiss.append({"Index": "IndexFlatIP (exact)", "nprobe": "-", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
print(f"Exact    MRR: {mrr:.4f}, {t*1000:.1f} ms/q")

# Approximate: IVFFlat with varying nprobe
nlist = 256
quantizer = faiss.IndexFlatIP(dim)
idx_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
idx_ivf.train(doc_embs)
idx_ivf.add(doc_embs)

for nprobe in [1, 4, 16, 64, 128]:
    idx_ivf.nprobe = nprobe
    r = make_ivf_retriever(idx_ivf)
    mrr = mrr_at_10(r, ds)
    t   = measure_retrieval_time(r, queries)
    results_faiss.append({"Index": f"IndexIVFFlat (nlist={nlist})", "nprobe": nprobe, "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"nprobe={nprobe:<4} MRR: {mrr:.4f}, {t*1000:.1f} ms/q")

del idx_flat, idx_ivf, doc_embs; gc.collect()
pd.DataFrame(results_faiss)

## 5. BM25 k1=0.5 Inside Hybrid

The distance metrics ablation found k1=0.5 gives better standalone BM25 MRR (0.37 vs 0.34).
Does this carry over when BM25 is used as a candidate retriever inside the Hybrid?

In [ ]:
results_k1 = []

for k1, label in [(1.5, "Hybrid (BM25 k1=1.5, default)"), (0.5, "Hybrid (BM25 k1=0.5, tuned)")]:
    hybrid = HybridRetriever(top_k=TOP_K, candidate_k=50)
    hybrid.bm25_retriever = BM25Retriever(top_k=50, k1=k1)
    hybrid.fit(passages_text)
    mrr = mrr_at_10(hybrid, ds)
    t   = measure_retrieval_time(hybrid, queries)
    results_k1.append({"Config": label, "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"{label}  MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del hybrid; gc.collect()

pd.DataFrame(results_k1)

## Summary

In [ ]:
print("=== 1. Candidate pool size ===")
print(pd.DataFrame(results_pool).to_string(index=False))
print()
print("=== 2. BM25 Tokenization ===")
print(pd.DataFrame(results_tok).to_string(index=False))
print()
print("=== 3. Query length ===")
print(pd.DataFrame(results_len).pivot(index="Bucket", columns="Retriever", values="MRR@10").to_string())
print()
print("=== 4. Approximate FAISS ===")
print(pd.DataFrame(results_faiss).to_string(index=False))
print()
print("=== 5. BM25 k1 in Hybrid ===")
print(pd.DataFrame(results_k1).to_string(index=False))